In [1]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# ============================================================
# FUNCTION 3 - WEEK 9 BAYESIAN OPTIMISATION
# Run from inside the week9/ folder
# ============================================================

# ------------------------------------------------------------
# 1. Load cumulative Week 9 data
# ------------------------------------------------------------

X = np.load("function3/initial_inputs.npy")
Y = np.load("function3/initial_outputs.npy").reshape(-1)

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(best_x, "->", best_y)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. Week 8 calibration check
# ------------------------------------------------------------
#
# Week 8 selected:
# [0.33234786, 0.55398733, 0.42064452]
#
# GP prediction:
# mean ≈ -0.007519
# std  ≈ 0.010054
#
# Actual:
# -0.0026301843171150554
# ------------------------------------------------------------

week8_pred_mean = -0.007519
week8_pred_std = 0.010054
week8_actual = -0.0026301843171150554

week8_error = week8_actual - week8_pred_mean
week8_z_error = week8_error / week8_pred_std

print("\n================================")
print("WEEK 8 CALIBRATION CHECK")
print("================================")

print("Predicted mean:", week8_pred_mean)
print("Predicted std :", week8_pred_std)
print("Actual        :", week8_actual)

print("\nPrediction error:")
print(week8_error)

print("\nError / predicted std:")
print(week8_z_error)


# ------------------------------------------------------------
# 3. Fit ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(1e-3, 1e3)
    )
    *
    Matern(
        length_scale=np.ones(3) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=20,
    random_state=42
)

gp.fit(X, Y)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(gp.kernel_)

lengthscales = gp.kernel_.k1.k2.length_scale

inverse_ls = 1.0 / lengthscales
relative_sensitivity = inverse_ls / inverse_ls.sum()

print("\nARD lengthscales:")
print(lengthscales)

print("\nNormalised inverse-lengthscale sensitivity:")
print(relative_sensitivity)


# ------------------------------------------------------------
# 4. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(mu, sigma, best_y, xi=0.0):

    improvement = mu - best_y - xi

    valid = sigma > 1e-12

    Z = np.zeros_like(mu)
    Z[valid] = improvement[valid] / sigma[valid]

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid] * norm.cdf(Z[valid])
        +
        sigma[valid] * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 5. Candidate generation
# ------------------------------------------------------------
#
# 3D is still small enough for strong coverage:
# local + wide + large global pool.
# ------------------------------------------------------------

rng = np.random.default_rng(42)

local_scale = np.clip(
    0.25 * lengthscales,
    0.015,
    0.10
)

wide_scale = np.clip(
    0.50 * lengthscales,
    0.04,
    0.20
)

print("\nLocal widths:", local_scale)
print("Wide widths:", wide_scale)

local_candidates = (
    best_x
    + rng.normal(
        0,
        local_scale,
        size=(70000, 3)
    )
)

wide_candidates = (
    best_x
    + rng.normal(
        0,
        wide_scale,
        size=(50000, 3)
    )
)

global_candidates = rng.uniform(
    0,
    1,
    size=(100000, 3)
)

local_candidates = np.clip(
    local_candidates,
    0,
    1
)

wide_candidates = np.clip(
    wide_candidates,
    0,
    1
)

candidates = np.vstack([
    local_candidates,
    wide_candidates,
    global_candidates
])

# Remove near-duplicates

tree = cKDTree(X)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

print("\nCandidates after duplicate filtering:")
print(len(candidates))


# ------------------------------------------------------------
# 6. GP predictions
# ------------------------------------------------------------

mu, sigma = gp.predict(
    candidates,
    return_std=True
)


# ------------------------------------------------------------
# 7. Primary EI
# ------------------------------------------------------------

EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(EI)

print("\n================================")
print("PRIMARY EI")
print("================================")

print("candidate =", candidates[ei_idx])
print("mean =", mu[ei_idx])
print("std =", sigma[ei_idx])
print("EI =", EI[ei_idx])


# ------------------------------------------------------------
# 8. EI sensitivity
# ------------------------------------------------------------

y_scale = np.std(Y)

xi_values = [
    0.0,
    0.01 * y_scale,
    0.05 * y_scale,
    0.10 * y_scale
]

print("\n================================")
print("EI SENSITIVITY")
print("================================\n")

for xi in xi_values:

    EI_test = expected_improvement(
        mu,
        sigma,
        best_y,
        xi
    )

    idx = np.argmax(EI_test)

    print(
        "xi =", f"{xi:.6e}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n EI =", round(EI_test[idx], 8),
        "\n"
    )


# ------------------------------------------------------------
# 9. Highest predicted mean
# ------------------------------------------------------------

mean_idx = np.argmax(mu)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print("candidate =", candidates[mean_idx])
print("mean =", mu[mean_idx])
print("std =", sigma[mean_idx])


# ------------------------------------------------------------
# 10. UCB diagnostic
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================\n")

for beta in [0.1, 0.25, 0.5, 1.0]:

    UCB = mu + beta * sigma
    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n UCB =", round(UCB[idx], 6),
        "\n"
    )

X shape: (23, 3)
Y shape: (23,)

Current best:
[0.332348 0.553987 0.420645] -> -0.00263018431711505

Y range:
min = -0.3989255131463011
max = -0.00263018431711505
std = 0.07741828084561575

WEEK 8 CALIBRATION CHECK
Predicted mean: -0.007519
Predicted std : 0.010054
Actual        : -0.0026301843171150554

Prediction error:
0.004888815682884944

Error / predicted std:
0.4862557870384866


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-08. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(



GP FIT

Fitted kernel:
1.69**2 * Matern(length_scale=[0.793, 1.29, 0.249], nu=2.5) + WhiteKernel(noise_level=1e-08)

ARD lengthscales:
[0.79311412 1.28712568 0.24853593]

Normalised inverse-lengthscale sensitivity:
[0.20801547 0.12817708 0.66380746]

Local widths: [0.1        0.1        0.06213398]
Wide widths: [0.2        0.2        0.12426797]

Candidates after duplicate filtering:
219916

PRIMARY EI
candidate = [2.42790427e-01 9.53648726e-01 6.38547696e-04]
mean = -0.016762675122445808
std = 0.05910129597405071
EI = 0.017182660651114477

EI SENSITIVITY

xi = 0.000000e+00 
 candidate = [2.42790427e-01 9.53648726e-01 6.38547696e-04] 
 mean = -0.016763 
 std = 0.059101 
 EI = 0.01718266 

xi = 7.741828e-04 
 candidate = [1.79427975e-01 9.98035015e-01 8.33538086e-04] 
 mean = -0.021687 
 std = 0.064122 
 EI = 0.01687949 

xi = 3.870914e-03 
 candidate = [1.79427975e-01 9.98035015e-01 8.33538086e-04] 
 mean = -0.021687 
 std = 0.064122 
 EI = 0.01573548 

xi = 7.741828e-03 
 candidate =

In [2]:
# ============================================================
# FINAL FUNCTION 3 - WEEK 9 SELECTION
# ============================================================
#
# Week 8 was well calibrated:
# realised error ≈ +0.49 predictive standard deviations.
#
# The highest GP mean and beta=0.1 UCB remain close to the
# successful Week 8 region.
#
# beta >= 0.25 causes an abrupt move toward x3 ~ 0 and x2 ~ 1,
# driven by much higher uncertainty.
#
# Therefore beta=0.1 is selected: it adds modest exploration
# while retaining the locally well-supported GP region.

beta = 0.1

UCB = mu + beta * sigma
final_idx = np.argmax(UCB)

week9_candidate = candidates[final_idx]

print("Week 9 Function 3 candidate:")
print(week9_candidate)

print("\nPredicted mean:")
print(mu[final_idx])

print("\nPredicted std:")
print(sigma[final_idx])

print("\nUCB:")
print(UCB[final_idx])

portal = "-".join(
    f"{x:.6f}"
    for x in week9_candidate
)

print("\nPortal format:")
print(portal)

Week 9 Function 3 candidate:
[0.32024933 0.6049606  0.41523034]

Predicted mean:
-0.0020063777706703922

Predicted std:
0.0021499303089734974

UCB:
-0.0017913847397730426

Portal format:
0.320249-0.604961-0.415230
